# Assignment 2 — Question 1: Used-car data preparation and manipulation

**Dataset:** `data_raw/train.csv`

**Target:** `Price` in lakhs

**Tasks:** (a) missing values; (b) unit removal; (c) one-hot encoding; (d) engineered feature; (e) select/filter/rename/mutate/arrange/grouped summaries.

Run cells top-to-bottom. All file locations are resolved relative to either repository root or `src/`. The original source file is never overwritten.

In [44]:
# ============================================================
# 1. Import Required Python Libraries
# ============================================================

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 2. Identify the Project Root Directory
# ============================================================

# Determine whether the notebook is running from the
# project root or from the src subdirectory.
current_directory = Path.cwd()

if (current_directory / "data_raw" / "train.csv").is_file():
    ROOT = current_directory

elif (current_directory.parent / "data_raw" / "train.csv").is_file():
    ROOT = current_directory.parent

else:
    raise FileNotFoundError(
        "Unable to locate data_raw/train.csv. "
        "Run the notebook from the project root or src folder."
    )


# ============================================================
# 3. Load the Used Cars Dataset
# ============================================================

# Define the location of the original used cars dataset.
dataset_path = ROOT / "data_raw" / "train.csv"

# Load the CSV file into a pandas DataFrame.
df = pd.read_csv(dataset_path)


# ============================================================
# 4. Validate Dataset Loading
# ============================================================

# Confirm that the dataset contains observations.
assert not df.empty, (
    "The used cars dataset is empty."
)

# Confirm that the target variable is present.
assert "Price" in df.columns, (
    "The required target variable 'Price' is missing."
)

# Confirm that the required preprocessing columns exist.
required_columns = [
    "Name",
    "Location",
    "Year",
    "Mileage",
    "Kilometers_Driven",
    "Fuel_Type",
    "Transmission",
    "Engine",
    "Power",
    "Seats",
    "New_Price",
    "Price"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)


# ============================================================
# 5. Display Dataset Information
# ============================================================

# Report the dataset location and dimensions.
print("USED CARS DATASET INFORMATION")
print("-" * 60)

print(f"Dataset: {dataset_path.relative_to(ROOT)}")
print(f"Number of Records: {len(df):,}")
print(f"Number of Columns: {len(df.columns)}")

# Display the first five observations.
print("\nPreview of Used Cars Dataset:")

display(df.head())

print("\nDataset loaded and validated successfully.")


USED CARS DATASET INFORMATION
------------------------------------------------------------
Dataset: data_raw/train.csv
Number of Records: 5,847
Number of Columns: 14

Preview of Used Cars Dataset:


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50



Dataset loaded and validated successfully.


## (a) Missing Values and Justification for Treatment

All columns were examined for missing values before performing data cleaning. The unnamed index column was removed because it represents a row identifier rather than a meaningful vehicle attribute.

Missing values in `Mileage`, `Engine`, `Power`, and `Seats` were replaced using the **median of each respective column**. Median imputation was selected because these numerical attributes may contain skewed distributions or extreme observations, making the median less sensitive to outliers than the mean. Additionally, the relatively small proportion of missing values in these columns makes median imputation a reasonable approach for preserving dataset completeness.

Measurement units were removed and numerical values extracted before imputation to ensure that replacement values were calculated using numeric data.

The `New_Price` attribute required special consideration because more than 80% of its observations were missing. Removing all records with missing `New_Price` values would substantially reduce the dataset. Therefore, missing values were replaced with the median of the available prices, and an additional binary variable, `New_Price_Missing`, was created to identify records whose original prices were unavailable.

**This treatment preserves the observations but does not recover the missing price information.** Because most `New_Price` values were unavailable, the imputed values should be interpreted as placeholders rather than reliable estimates of original vehicle prices. Consequently, this attribute should not be used to draw substantive conclusions about original vehicle pricing without additional investigation.

The target variable, `Price`, contained no missing values and was preserved without modification. If target values had been missing, those observations would have been excluded rather than assigning artificial target values.

Overall, the selected approach preserves the dataset's observations, handles missing values consistently, and documents the limitations associated with extensive missingness in `New_Price`.

In [45]:
# ============================================================
# 1. Calculate Missing Values by Column
# ============================================================

# Count the number of missing observations in each column.
missing_counts = df.isna().sum()

# Calculate the percentage of missing observations
# relative to the total number of dataset records.
missing_percentages = df.isna().mean() * 100


# ============================================================
# 2. Create a Missing Value Summary Table
# ============================================================

# Combine missing-value counts and percentages
# into a single DataFrame for easier analysis.
missing_before = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percent": missing_percentages
})

# Sort columns by the number of missing values,
# placing those with the most missing values first.
missing_before = missing_before.sort_values(
    by="Missing_Count",
    ascending=False
)


# ============================================================
# 3. Display Missing Value Statistics
# ============================================================

print("MISSING VALUE ANALYSIS")
print("-" * 60)

# Display missing-value counts and percentages.
display(
    missing_before.round(2)
)

# Calculate the total number of missing values.
total_missing_values = int(
    missing_counts.sum()
)

# Count the number of columns containing missing values.
columns_with_missing_values = int(
    (missing_counts > 0).sum()
)

print(f"\nTotal Missing Values: {total_missing_values:,}")

print(
    f"Columns Containing Missing Values: "
    f"{columns_with_missing_values}"
)


# ============================================================
# 4. Identify Duplicate Records
# ============================================================

# Count fully duplicated rows across all dataset columns.
duplicate_record_count = int(
    df.duplicated().sum()
)

print("\nDUPLICATE RECORD ANALYSIS")
print("-" * 60)

print(
    f"Number of Duplicate Records: "
    f"{duplicate_record_count:,}"
)

# Calculate the percentage of duplicate records.
duplicate_percentage = (
    duplicate_record_count / len(df)
) * 100

print(
    f"Percentage of Duplicate Records: "
    f"{duplicate_percentage:.2f}%"
)


# ============================================================
# 5. Validate Data Quality Calculations
# ============================================================

# Confirm that missing-value counts are nonnegative.
assert (missing_before["Missing_Count"] >= 0).all(), (
    "Invalid negative missing-value counts detected."
)

# Confirm that missing percentages fall between 0 and 100.
assert missing_before["Missing_Percent"].between(
    0, 100
).all(), (
    "Missing-value percentages are outside the valid range."
)

# Confirm that duplicate counts cannot exceed the
# number of observations in the dataset.
assert 0 <= duplicate_record_count < len(df), (
    "Invalid duplicate record count detected."
)

print("\nData quality analysis completed successfully.")


MISSING VALUE ANALYSIS
------------------------------------------------------------


,Missing_Count,Missing_Percent
New_Price,5032,86.06
Seats,38,0.65
Power,36,0.62
Engine,36,0.62
Mileage,2,0.03
Unnamed: 0,0,0.00
Fuel_Type,0,0.00
Kilometers_Driven,0,0.00
Year,0,0.00
Location,0,0.00



Total Missing Values: 5,144
Columns Containing Missing Values: 5

DUPLICATE RECORD ANALYSIS
------------------------------------------------------------
Number of Duplicate Records: 0
Percentage of Duplicate Records: 0.00%

Data quality analysis completed successfully.


## (b) Clean Numeric Attributes and Standardize Units

The `Mileage`, `Engine`, `Power`, and `New_Price` attributes originally contained numerical measurements combined with text-based measurement units. These attributes were converted into numerical values to support subsequent data cleaning and statistical analysis.

- **Mileage:** Removed the `kmpl` and `km/kg` unit labels and extracted the numerical values. Because kilometers per liter and kilometers per kilogram represent different fuel-efficiency measurements, the original units were preserved in a separate `Mileage_Unit` column. This prevents losing important information about how each measurement was originally expressed.

- **Engine:** Removed the `CC` label and converted engine displacement into numerical values, retaining cubic centimeters as the implied measurement unit.

- **Power:** Removed the `bhp` label and converted engine power into numerical values, retaining brake horsepower as the implied measurement unit.

- **New_Price:** Extracted numerical prices and standardized all values to **lakhs**. Prices originally expressed in crores (`Cr`) were multiplied by 100 because one crore equals 100 lakhs. Prices already expressed in lakhs remained unchanged.

These transformations ensure that the four attributes contain numeric values rather than strings with embedded units. Standardizing `New_Price` to a common monetary unit also makes price comparisons consistent.

**Limitation:** Although the mileage unit labels were removed, `kmpl` and `km/kg` values were not converted into a common fuel-efficiency measurement. Consequently, mileage values with different original units should not be directly compared or aggregated without accounting for `Mileage_Unit`.

In [46]:
# ============================================================
# 1. Create a Working Copy of the Dataset
# ============================================================

# Remove the unnecessary index column if it exists.
clean = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()


# ============================================================
# 2. Extract Mileage Units
# ============================================================

# Preserve the original mileage units before converting
# the Mileage column to numeric values.
clean["Mileage_Unit"] = (
    clean["Mileage"]
    .astype("string")
    .str.extract(r"(kmpl|km/kg)", expand=False)
    .fillna("Unknown")
)


# ============================================================
# 3. Convert Vehicle Attributes to Numeric Values
# ============================================================

# Remove units such as kmpl, CC, and bhp.
numeric_columns = ["Mileage", "Engine", "Power"]

for column in numeric_columns:
    clean[column] = pd.to_numeric(
        clean[column]
        .astype("string")
        .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False),
        errors="coerce"
    )


# ============================================================
# 4. Clean and Standardize New Car Prices
# ============================================================

# Preserve the original price strings.
original_new_price = clean["New_Price"].astype("string")

# Extract numerical price values.
new_price_values = pd.to_numeric(
    original_new_price.str.extract(
        r"([0-9]+(?:\.[0-9]+)?)",
        expand=False
    ),
    errors="coerce"
)

# Identify the original price units.
new_price_units = (
    original_new_price
    .str.extract(r"(?i)\b(Lakh|Cr)\b", expand=False)
    .str.lower()
)

# Convert crore values to lakhs (1 crore = 100 lakhs).
clean["New_Price"] = np.where(
    new_price_units.eq("cr").fillna(False),
    new_price_values * 100,
    new_price_values
)


# ============================================================
# 5. Identify and Handle Missing Values
# ============================================================

# Create an indicator for originally missing new prices.
clean["New_Price_Missing"] = (
    clean["New_Price"].isna().astype("int8")
)

# Record the number of missing new prices.
new_price_missing_before = int(
    clean["New_Price_Missing"].sum()
)

# Replace missing new prices with the median.
clean["New_Price"] = clean["New_Price"].fillna(
    clean["New_Price"].median()
)

# Replace missing numeric values with column medians.
columns_to_impute = ["Mileage", "Engine", "Power", "Seats"]

for column in columns_to_impute:
    clean[column] = clean[column].fillna(
        clean[column].median()
    )


# ============================================================
# 6. Validate the Cleaned Dataset
# ============================================================

# Confirm that all new car prices are nonnegative.
assert clean["New_Price"].dropna().ge(0).all(), (
    "New_Price contains negative values."
)

# Confirm that no missing values remain.
assert clean.isna().sum().sum() == 0, (
    "Missing values remain in the cleaned dataset."
)


# ============================================================
# 7. Display Cleaning Results
# ============================================================

print("Missing Values After Cleaning:")

display(
    clean.isna().sum().to_frame(name="Missing_Count")
)

print("\nPreview of Cleaned Numeric Attributes:")

display(
    clean[
        [
            "Mileage",
            "Engine",
            "Power",
            "Seats",
            "New_Price",
            "Mileage_Unit"
        ]
    ].head(8)
)

print(
    "\nMissing New_Price Values Replaced:",
    new_price_missing_before
)


Missing Values After Cleaning:


,Missing_Count
Name,0
Location,0
Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,0
Engine,0
Power,0



Preview of Cleaned Numeric Attributes:


,Mileage,Engine,Power,Seats,New_Price,Mileage_Unit
0,19.67,1582,126.2,5.0,11.75,kmpl
1,13.0,1199,88.7,5.0,8.61,km/kg
2,20.77,1248,88.76,7.0,11.75,kmpl
3,15.2,1968,140.8,5.0,11.75,kmpl
4,23.08,1461,63.1,5.0,11.75,kmpl
5,11.36,2755,171.5,8.0,21.00,kmpl
6,20.54,1598,103.6,5.0,11.75,kmpl
7,22.3,1248,74.0,5.0,11.75,kmpl



Missing New_Price Values Replaced: 5032


## (c) One-hot encode Fuel_Type and Transmission

Use separate 0/1 indicators for every observed category (`drop_first=False`), preserving all information. If the encoded data are later used in a linear regression with an intercept, one indicator per group would normally be removed to prevent perfect multicollinearity.

In [47]:
# ============================================================
# 1. Identify Categorical Columns
# ============================================================

# Specify the categorical attributes to encode.
categorical_columns = [
    "Fuel_Type",
    "Transmission"
]


# ============================================================
# 2. Perform One-Hot Encoding
# ============================================================

# Convert each categorical value into a separate binary column.
# Retain all categories and use integers (0 and 1).
encoded = pd.get_dummies(
    clean,
    columns=categorical_columns,
    prefix=categorical_columns,
    dtype=int,
    drop_first=False
)


# ============================================================
# 3. Identify the Newly Encoded Columns
# ============================================================

# Extract the names of the generated indicator columns.
encoded_columns = [
    column
    for column in encoded.columns
    if column.startswith(("Fuel_Type_", "Transmission_"))
]

# Separate the indicators by their original attributes.
fuel_columns = [
    column
    for column in encoded_columns
    if column.startswith("Fuel_Type_")
]

transmission_columns = [
    column
    for column in encoded_columns
    if column.startswith("Transmission_")
]


# ============================================================
# 4. Validate the One-Hot Encoding
# ============================================================

# Each vehicle must have exactly one active fuel type.
assert (encoded[fuel_columns].sum(axis=1) == 1).all(), (
    "Some vehicles have invalid fuel type encoding."
)

# Each vehicle must have exactly one active transmission type.
assert (encoded[transmission_columns].sum(axis=1) == 1).all(), (
    "Some vehicles have invalid transmission encoding."
)


# ============================================================
# 5. Display Encoding Results
# ============================================================

print("One-Hot Encoded Columns:")

for column in encoded_columns:
    print(f"  - {column}")

print("\nPreview of Encoded Categorical Attributes:")

display(
    encoded[encoded_columns].head()
)

print("\nOne-hot encoding validation completed successfully.")


One-Hot Encoded Columns:
  - Fuel_Type_Diesel
  - Fuel_Type_Electric
  - Fuel_Type_Petrol
  - Transmission_Automatic
  - Transmission_Manual

Preview of Encoded Categorical Attributes:


,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
0,1,0,0,0,1
1,0,0,1,0,1
2,1,0,0,0,1
3,1,0,0,1,0
4,1,0,0,0,1



One-hot encoding validation completed successfully.


## (d) Engineer car age

Age is computed using the **execution year**, not a hard-coded calendar year, so the notebook remains reproducible in future years (the analysis date is reported). Source manufacturing year is preserved.

In [48]:
# ============================================================
# 1. Determine the Current Analysis Year
# ============================================================

# Retrieve the current calendar year for the analysis.
analysis_year = datetime.now().year


# ============================================================
# 2. Create the Car Age Feature
# ============================================================

# Calculate each vehicle's age by subtracting its
# manufacturing year from the current year.
encoded["Car_Age"] = analysis_year - encoded["Year"]


# ============================================================
# 3. Validate the New Feature
# ============================================================

# Verify that no vehicle has a negative calculated age.
# Negative values may indicate incorrect manufacturing years.
assert (encoded["Car_Age"] >= 0).all(), (
    "Negative car ages detected. "
    "Inspect future-dated manufacturing years."
)


# ============================================================
# 4. Display Feature Engineering Results
# ============================================================

print(f"Analysis Year: {analysis_year}")

print("\nPreview of Vehicle Ages:")

display(
    encoded[
        [
            "Name",
            "Year",
            "Car_Age"
        ]
    ].head()
)

print("\nCar age feature created and validated successfully.")


Analysis Year: 2026

Preview of Vehicle Ages:


,Name,Year,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,2015,11
1,Honda Jazz V,2011,15
2,Maruti Ertiga VDI,2012,14
3,Audi A4 New 2.0 TDI Multitronic,2013,13
4,Nissan Micra Diesel XV,2013,13



Car age feature created and validated successfully.


## (e) Python equivalents: select, filter, rename, mutate, arrange, and group-by summarize

The examples are separate transformations of `encoded`, so each operation can be evaluated independently.

In [49]:
# ============================================================
# 1. SELECT: Choose Relevant Dataset Attributes
# ============================================================

# Select the columns most relevant to vehicle pricing
# and age analysis.
selected_columns = [
    "Name",
    "Location",
    "Year",
    "Car_Age",
    "Price"
]

selected = encoded[selected_columns].copy()

# Display the selected attributes.
print("SELECT: Relevant Vehicle Attributes")

display(selected.head())


# ============================================================
# 2. FILTER: Apply Conditions to Vehicle Listings
# ============================================================

# Retain vehicles that meet both conditions:
# 1. Used car price is at least 10 lakhs.
# 2. Vehicle age is 20 years or less.
filtered = encoded.loc[
    (encoded["Price"] >= 10) &
    (encoded["Car_Age"] <= 20)
].copy()

# Display the number of matching vehicles and a preview.
print("\nFILTER: Vehicles Meeting Price and Age Criteria")

print(f"Number of Matching Vehicles: {len(filtered)}")

display(
    filtered[
        [
            "Name",
            "Price",
            "Car_Age"
        ]
    ].head()
)


# ============================================================
# 3. RENAME: Assign More Descriptive Column Names
# ============================================================

# Rename the used car price and odometer columns
# to explicitly identify their measurement units.
renamed = encoded.rename(
    columns={
        "Price": "Used_Price_Lakh",
        "Kilometers_Driven": "Odometer_Km"
    }
)

# Display the renamed attributes.
print("\nRENAME: Descriptive Column Names")

display(
    renamed[
        [
            "Used_Price_Lakh",
            "Odometer_Km"
        ]
    ].head()
)


# ============================================================
# 4. MUTATE: Calculate Price per 1,000 Kilometers
# ============================================================

# Create a new feature measuring the vehicle's listed
# price in lakhs per 1,000 kilometers driven.
#
# Prevent division by zero by assigning NaN when
# the odometer reading is zero or negative.
mutated = encoded.assign(
    Price_Per_1000_Km=lambda data: np.where(
        data["Kilometers_Driven"] > 0,
        (data["Price"] * 1000) / data["Kilometers_Driven"],
        np.nan
    )
)

# Display the original and calculated attributes.
print("\nMUTATE: Price per 1,000 Kilometers")

display(
    mutated[
        [
            "Price",
            "Kilometers_Driven",
            "Price_Per_1000_Km"
        ]
    ].head()
)


# ============================================================
# 5. ARRANGE: Sort Vehicle Listings
# ============================================================

# Sort vehicles according to:
# 1. Highest used car price first.
# 2. Newest manufacturing year when prices are equal.
arranged = encoded.sort_values(
    by=["Price", "Year"],
    ascending=[False, False]
)

# Display the highest-priced vehicles.
print("\nARRANGE: Vehicles Sorted by Price and Year")

display(
    arranged[
        [
            "Name",
            "Year",
            "Price"
        ]
    ].head()
)


# ============================================================
# 6. GROUP BY: Reconstruct Transmission Categories
# ============================================================

# Identify the transmission indicator columns generated
# during the previous one-hot encoding operation.
transmission_columns = [
    column
    for column in encoded_columns
    if column.startswith("Transmission_")
]

# Build conditions identifying the active transmission type.
transmission_conditions = [
    encoded[column].eq(1)
    for column in transmission_columns
]

# Extract the original transmission category names.
transmission_labels = [
    column.replace("Transmission_", "")
    for column in transmission_columns
]

# Reconstruct readable transmission categories
# from the one-hot encoded indicator columns.
transmission_categories = np.select(
    transmission_conditions,
    transmission_labels,
    default="Other"
)

# Add the reconstructed category to a working dataset.
summary_input = encoded.assign(
    Transmission_Category=transmission_categories
)


# ============================================================
# 7. SUMMARIZE: Calculate Statistics by Transmission Type
# ============================================================

# Group vehicle listings by transmission category
# and calculate descriptive statistics.
grouped = (
    summary_input
    .groupby(
        "Transmission_Category",
        dropna=False
    )
    .agg(
        Listings=("Price", "size"),
        Mean_Price_Lakh=("Price", "mean"),
        Median_Price_Lakh=("Price", "median"),
        Mean_Age=("Car_Age", "mean")
    )
    .round(2)
    .sort_values(
        by="Mean_Price_Lakh",
        ascending=False
    )
)

# Display the aggregated results.
print("\nGROUP BY + SUMMARIZE: Transmission Statistics")

display(grouped)


# ============================================================
# 8. Validate Data Manipulation Results
# ============================================================

# Verify that selecting columns preserves the row count.
assert len(selected) == len(encoded), (
    "SELECT unexpectedly changed the number of rows."
)

# Confirm that all filtered vehicles meet the conditions.
assert (
    (filtered["Price"] >= 10) &
    (filtered["Car_Age"] <= 20)
).all(), (
    "FILTER contains vehicles outside the specified criteria."
)

# Confirm that the new calculated attribute exists.
assert "Price_Per_1000_Km" in mutated.columns, (
    "The calculated price-per-distance feature is missing."
)

# Confirm that all original records are represented
# in the grouped summary.
assert grouped["Listings"].sum() == len(encoded), (
    "GROUP BY results do not account for all vehicle listings."
)

print("\nAll data manipulation operations validated successfully.")


SELECT: Relevant Vehicle Attributes


,Name,Location,Year,Car_Age,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,11,12.50
1,Honda Jazz V,Chennai,2011,15,4.50
2,Maruti Ertiga VDI,Chennai,2012,14,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,13,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,13,3.50



FILTER: Vehicles Meeting Price and Age Criteria
Number of Matching Vehicles: 1493


,Name,Price,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,12.50,11
3,Audi A4 New 2.0 TDI Multitronic,17.74,13
5,Toyota Innova Crysta 2.8 GX AT 8S,17.50,10
11,Land Rover Range Rover 2.2L Pure,27.00,12
12,Land Rover Freelander 2 TD4 SE,17.50,14



RENAME: Descriptive Column Names


,Used_Price_Lakh,Odometer_Km
0,12.50,41000
1,4.50,46000
2,6.00,87000
3,17.74,40670
4,3.50,86999


... (3 outputs left out)

,Name,Year,Price
3952,Land Rover Range Rover 3.0 Diesel LWB Vogue,2017,160.00
5620,Lamborghini Gallardo Coupe,2011,120.00
5752,Jaguar F Type 5.0 V8 S,2015,100.00
1457,Land Rover Range Rover Sport SE,2019,97.07
1917,BMW 7 Series 740Li,2018,93.67



GROUP BY + SUMMARIZE: Transmission Statistics


,Listings,Mean_Price_Lakh,Median_Price_Lakh,Mean_Age
Transmission_Category,,,,
Automatic,1712,19.89,16.00,12.12
Manual,4135,5.42,4.65,12.73



All data manipulation operations validated successfully.


## Export and Interpretation

The final cleaned dataset was exported to `data_clean/used_cars_cleaned.csv` for reproducibility and future analysis. The exported dataset preserves the original vehicle records and the target variable, `Price`, without modifying the observed used-car prices.

The dataset includes numerical versions of `Mileage`, `Engine`, `Power`, and `New_Price`, one-hot encoded categorical variables for `Fuel_Type` and `Transmission`, and the newly created `Car_Age` feature.

Missing numerical values were handled using median imputation. Because `New_Price` contained substantial missing data, a binary `New_Price_Missing` indicator was retained to distinguish originally observed prices from imputed values. The `Mileage_Unit` column was also preserved because `kmpl` and `km/kg` represent different fuel-efficiency measurements.

Validation checks confirmed that the original number of observations was preserved, the target variable remained unchanged, and the required numerical attributes contained no missing values after preprocessing.

**Overall, the exported dataset is suitable for subsequent descriptive analysis, provided that the limitations of imputed `New_Price` values and differing mileage units are considered.** All statistical summaries are descriptive and should not be interpreted as evidence of causal relationships.

In [50]:
# ============================================================
# 1. Define the Output File Path
# ============================================================

# Specify the destination for the cleaned dataset.
# The file will be saved in the project's data_clean folder.
output_path = ROOT / "data_clean" / "used_cars_cleaned.csv"

# Ensure that the output directory exists.
output_path.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Save the Cleaned Dataset
# ============================================================

# Export the cleaned and encoded dataset to a CSV file.
# Exclude the pandas index to avoid an unnecessary column.
encoded.to_csv(
    output_path,
    index=False
)

print("Cleaned Dataset Saved Successfully")

print(f"File Location: {output_path.relative_to(ROOT)}")
print(f"Total Rows: {len(encoded):,}")
print(f"Total Columns: {len(encoded.columns)}")


# ============================================================
# 3. Report Original Missing Value Statistics
# ============================================================

# Calculate the percentage of New_Price values that
# were missing before median imputation.
new_price_missing_percentage = (
    encoded["New_Price_Missing"].mean() * 100
)

print("\nOriginal Missing Value Statistics")

print(
    f"New_Price Missing Percentage: "
    f"{new_price_missing_percentage:.1f}%"
)


# ============================================================
# 4. Compare Original and Cleaned Target Statistics
# ============================================================

# Calculate the mean used car price before cleaning.
original_mean_price = df["Price"].mean()

# Calculate the mean used car price after cleaning.
cleaned_mean_price = encoded["Price"].mean()

print("\nUsed Car Price Comparison")

print(f"Original Mean Price (Lakhs): {original_mean_price:.3f}")
print(f"Cleaned Mean Price (Lakhs): {cleaned_mean_price:.3f}")


# ============================================================
# 5. Validate Dataset Integrity
# ============================================================

# Confirm that cleaning preserved the number of records.
assert len(encoded) == len(df), (
    "Row count mismatch between original and cleaned datasets."
)

# Confirm that the target variable was not modified.
assert encoded["Price"].equals(df["Price"]), (
    "The target variable changed during data cleaning."
)

# Confirm that the required numeric attributes
# contain no missing values.
required_numeric_columns = [
    "Mileage",
    "Engine",
    "Power",
    "Seats"
]

assert encoded[required_numeric_columns].notna().all().all(), (
    "Missing values remain in required numeric attributes."
)

# Confirm that the exported CSV file exists.
assert output_path.is_file(), (
    "The cleaned dataset was not saved successfully."
)

print("\nAll dataset integrity checks passed successfully.")


Cleaned Dataset Saved Successfully
File Location: data_clean/used_cars_cleaned.csv
Total Rows: 5,847
Total Columns: 19

Original Missing Value Statistics
New_Price Missing Percentage: 86.1%

Used Car Price Comparison
Original Mean Price (Lakhs): 9.654
Cleaned Mean Price (Lakhs): 9.654

All dataset integrity checks passed successfully.
